# Documentation de la classe `WeatherLoader`

La classe `WeatherLoader` (`kadi.weather.data.WeatherLoader`) est responsable de l'acquisition, de la mise en cache (via SQLite global dans KadiPy) et de la normalisation des données météorologiques. Elle gère la transition transparente entre les requêtes en temps réel (API Open-Meteo) et les bases de données historiques locales (CHIRPS).

## 1. Initialisation
L'initialisation requiert uniquement une instance de la classe `Location`.

In [ ]:
from kadi.weather.location import Location
from kadi.weather.data import WeatherLoader


loc = Location(lat=9.3333, lon=2.6333, name='Parakou')
weather = WeatherLoader(loc)

print(f"Source des données à l'initialisation : {weather.source}")

## 2. Récupération des Prévisions (`get_forecast`)
**Méthode :** `get_forecast(days: int = 7, refresh: bool = False) -> pd.DataFrame`

Cette méthode tente de charger d'abord le cache local. Si le cache a expiré ou si `refresh=True` est utilisé, elle appellera l'API d'Open-Meteo pour récupérer des prévisions allant jusqu'à `days` jours.
Elle retourne un DataFrame `pandas` indexé par `date`.

In [ ]:
# Récupération des prévisions sur 3 jours
forecast_df = weather.get_forecast(days=3)

print(forecast_df.shape)

print(f"Source : {weather.source}")
display(forecast_df.head())

## 3. Récupération de l'Historique (`get_historical`)
**Méthode :** `get_historical(months: int = 120, refresh: bool = False, source: str = None) -> pd.DataFrame`

Récupère l'historique météo. Le comportement technique de cette méthode est une aggrégation :
- Les données de **température** sont extraites d'Open-Meteo Historical.
- Les données de **précipitations** peuvent provenir d’Open-Meteo, de **CHIRPS** ou des deux, selon le paramètre `source`.

Ces données combinées sont ensuite sauvegardées dans le cache SQLite `weather_data` de KadiPy.

In [ ]:
# Historique sur 2 mois (approx. 60 jours)
historical_df = weather.get_historical(months=2)

print(historical_df.shape)

print(f"Source de l'historique : {weather.source}")
display(historical_df.head())

## 4. Architecture Interne (Méthodes Privées)
En cas de développement sur ce module, voici les méthodes privées gérant la plomberie interne :
- `_fetch_forecast` et `_fetch_historical` : Réalisent les appels réseaux avec un mécanisme de *retry* configuré (tentatives et backoff) via `kadi._utils.network`.
- `_from_cache(start, end)` : Lit les données SQLite.
- `_to_cache(data, kind)` : Exécute des `UPSERT` massifs dans SQLite pour rafraichir le cache.
- `_normalize` : S'assure que les dates sont bien formatées et utilisées comme index DataFrame.